In [1]:
# Michael Logos Sinuraya_202431094

# ============================================================
# 1. IMPORT LIBRARY
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Library berhasil diimpor.")


# ============================================================
# 2. LOAD DATASET
# ============================================================

df = pd.read_csv("heart.csv")

print("\nDataset berhasil dimuat.")
print("Jumlah baris dan kolom:", df.shape)

display(df.head())

print("\nInformasi dataset:")
df.info()

print("\nStatistik deskriptif:")
display(df.describe())


# ============================================================
# 3. MEMERIKSA TARGET DAN KELAS
# ============================================================

print("\nDistribusi kelas HeartDisease:")
display(df["HeartDisease"].value_counts())

print("\nDistribusi kelas dalam persentase:")
display(df["HeartDisease"].value_counts(normalize=True) * 100)

# 0 = Tidak terindikasi penyakit jantung
# 1 = Terindikasi penyakit jantung

plt.figure(figsize=(6,4))
df["HeartDisease"].value_counts().sort_index().plot(kind="bar")

plt.title("Distribusi Kelas Heart Disease")
plt.xlabel("HeartDisease (0 = Tidak, 1 = Ya)")
plt.ylabel("Jumlah Data")
plt.xticks(rotation=0)
plt.show()


# ============================================================
# 4. PEMBERSIHAN DAN PERSIAPAN DATA
# ============================================================

print("\nNilai kosong sebelum pembersihan:")
display(df.isna().sum())

print("\nJumlah data duplikat:", df.duplicated().sum())

# Menghapus data duplikat
df = df.drop_duplicates().copy()

X = df.drop(columns=["HeartDisease"])
y = df["HeartDisease"]

# Identifikasi fitur numerik dan kategorikal
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("\nFitur numerik:")
print(numeric_features)

print("\nFitur kategorikal:")
print(categorical_features)

print("\nNilai kosong setelah pembersihan duplikat:")
display(df.isna().sum())

print("\nJumlah data setelah pembersihan:", df.shape)


# ============================================================
# 5. MEMBAGI DATA LATIH DAN DATA UJI
# ============================================================

X = df.drop(columns=["HeartDisease"])
y = df["HeartDisease"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=0,
    stratify=y
)

print("\nJumlah data latih:", X_train.shape[0])
print("Jumlah data uji:", X_test.shape[0])

print("\nDistribusi kelas data latih:")
display(y_train.value_counts())

print("\nDistribusi kelas data uji:")
display(y_test.value_counts())


# ============================================================
# 6. MEMERIKSA RENTANG FITUR DATA LATIH
# ============================================================

print("\nMinimum fitur numerik:")
display(X_train[numeric_features].min())

print("\nMaksimum fitur numerik:")
display(X_train[numeric_features].max())

print("\nRentang fitur numerik:")
display(
    X_train[numeric_features].max() -
    X_train[numeric_features].min()
)


# ============================================================
# 7. PIPELINE PREPROCESSING DAN KNN
# ============================================================

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

pipeline_knn = Pipeline([
    ("preprocessor", preprocessor),
    ("knn", KNeighborsClassifier())
])

print("\nPipeline preprocessing dan KNN berhasil dibuat.")


# ============================================================
# 8. GRIDSEARCHCV UNTUK MENENTUKAN k TERBAIK
# ============================================================

param_grid = {
    "knn__n_neighbors": list(range(1, 20))
}

grid_search = GridSearchCV(
    estimator=pipeline_knn,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("\nNilai k terbaik:")
print(grid_search.best_params_)

print("\nSkor validasi silang terbaik:")
print(grid_search.best_score_)


# ============================================================
# 9. MODEL TERBAIK DAN PREDIKSI DATA UJI
# ============================================================

model_knn = grid_search.best_estimator_

y_pred = model_knn.predict(X_test)

print("\nModel KNN berhasil dilatih.")
print("Prediksi data uji berhasil dilakukan.")


# ============================================================
# 10. EVALUASI MODEL
# ============================================================

accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)

plt.figure(figsize=(6,5))
plt.imshow(cm, interpolation="nearest")

plt.title("Confusion Matrix KNN")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")

plt.xticks([0,1], ["0", "1"])
plt.yticks([0,1], ["0", "1"])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i,j], ha="center", va="center")

plt.colorbar()
plt.show()


# ============================================================
# 11. UJI SATU DATA BARU
# ============================================================

# Mengambil satu baris dari data latih sebagai contoh data baru
data_baru = X_train.iloc[[0]].copy()

print("\nData baru yang diuji:")
display(data_baru)

# Memeriksa rentang fitur numerik berdasarkan data latih
rentang_min = X_train[numeric_features].min()
rentang_max = X_train[numeric_features].max()

validasi_rentang = (
    (data_baru[numeric_features] >= rentang_min) &
    (data_baru[numeric_features] <= rentang_max)
)

print("\nPemeriksaan rentang fitur numerik:")
display(validasi_rentang)

print("\nSemua fitur numerik berada dalam rentang data latih:")
print(validasi_rentang.all().all())

# Prediksi data baru
prediksi_baru = model_knn.predict(data_baru)

print("\nHasil prediksi data baru:", prediksi_baru[0])

if prediksi_baru[0] == 0:
    print("Prediksi kelas: 0 (Tidak terindikasi)")
else:
    print("Prediksi kelas: 1 (Terindikasi)")


# ============================================================
# 12. VISUALISASI SKOR CROSS VALIDATION TERHADAP k
# ============================================================

hasil_cv = pd.DataFrame(grid_search.cv_results_)

plt.figure(figsize=(8,5))

plt.plot(
    hasil_cv["param_knn__n_neighbors"].astype(int),
    hasil_cv["mean_test_score"],
    marker="o"
)

plt.title("Skor Cross Validation KNN terhadap Nilai k")
plt.xlabel("Nilai k")
plt.ylabel("Mean Accuracy Cross Validation")

plt.xticks(range(1,20))
plt.grid(True)
plt.show()

print("\nPraktikum KNN selesai.")

Library berhasil diimpor.


FileNotFoundError: [Errno 2] No such file or directory: 'heart.csv'